# Camera frame testbench

Interactive viewer for **Intel RealSense R435** RGB + depth synced to radar/lidar pairs.

1. Set `DATASET` or edit paths below.
2. Run **Build camera index** once (scans ROS bag header timestamps; slow on USB).
3. Run **Attach camera to sync** if `sync_pairs.csv` has no `camera_color_idx` column yet.
4. Scrub by **sync pair** index — loads color + depth for that matched moment.

Camera data lives under `{data_root}/camera/*.bag` (ROS bag v2 with RealSense topics).
Sync uses image `header.stamp` (Unix time), matched to each row's `radar_t`.

In [ ]:
from pathlib import Path

DATASET = "2026.05.10/18-05-08"

CAMERA_BAG = None
CAMERA_INDEX = None
SYNC_CSV = Path("res/2026.05.10/18-05-08/sync_pairs.csv")

PAIR_IDX = 0
DEPTH_MAX_M = 8.0

if DATASET:
    import sys

    _nb_root = Path.cwd()
    _sync_dir = _nb_root if (_nb_root / "datasets.json").is_file() else _nb_root / "sync"
    if str(_sync_dir) not in sys.path:
        sys.path.insert(0, str(_sync_dir))
    from dataset_config import load_dataset_paths

    paths = load_dataset_paths(DATASET)
    print(f"dataset: {DATASET}")
    CAMERA_BAG = paths.get("camera_bag")
    CAMERA_INDEX = paths.get("camera_index")
    SYNC_CSV = Path(paths.get("sync_csv", SYNC_CSV))

print("Camera bag:", CAMERA_BAG)
print("Camera index:", CAMERA_INDEX)
print("Sync CSV:", SYNC_CSV)

In [ ]:
# Build camera index (run once; rescans ROS bag)
from camera_compat import build_camera_index, load_camera_index

REBUILD_INDEX = False

index = load_camera_index(
    CAMERA_INDEX,
    bag_path=CAMERA_BAG,
    rebuild=REBUILD_INDEX,
)
print(f"color frames: {len(index['color_ts'])}")
print(f"depth frames: {len(index['depth_ts'])}")

In [ ]:
# Attach camera columns to sync_pairs.csv (skip if already present)
import csv
from sync_camera_pairs import attach_camera_columns, write_camera_csv

with open(SYNC_CSV, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames or []
    rows = list(reader)

if "camera_color_idx" not in fieldnames:
    rows = attach_camera_columns(rows, color_ts=index["color_ts"], depth_ts=index["depth_ts"], max_delta_ms=50.0)
    write_camera_csv(rows, SYNC_CSV)
    print("Updated", SYNC_CSV, "with camera columns")
else:
    print("Sync CSV already has camera columns")

In [ ]:
# View RGB + depth for one sync pair
import matplotlib.pyplot as plt
from sync_viz_data import load_sync_pairs_with_camera
from camera_compat import (
    DEFAULT_COLOR_TOPIC,
    DEFAULT_DEPTH_TOPIC,
    depth_to_display,
    read_image_at_index,
    rgb_to_display,
)

(
    _r,
    _l,
    delta_ms,
    radar_t,
    _lt,
    camera_color_idx,
    camera_depth_idx,
    camera_delta_ms,
) = load_sync_pairs_with_camera(SYNC_CSV)

pair = int(PAIR_IDX)
cidx = int(camera_color_idx[pair])
didx = int(camera_depth_idx[pair])
if cidx < 0:
    raise RuntimeError(f"Pair {pair} has no camera match. Re-run attach cell or widen max_delta_ms.")

print(
    f"pair={pair} radar_t={radar_t[pair]:.3f} lidar_delta={delta_ms[pair]:.1f} ms "
    f"camera_color_idx={cidx} camera_delta={camera_delta_ms[pair]:.1f} ms"
)

color = read_image_at_index(CAMERA_BAG, DEFAULT_COLOR_TOPIC, cidx)
depth = read_image_at_index(CAMERA_BAG, DEFAULT_DEPTH_TOPIC, didx)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(rgb_to_display(color.data))
axes[0].set_title(f"RGB #{cidx} ({color.width}x{color.height})")
axes[0].axis("off")

im = axes[1].imshow(depth_to_display(depth.data, max_m=DEPTH_MAX_M), cmap="turbo")
axes[1].set_title(f"Depth #{didx} (mm, max {DEPTH_MAX_M} m)")
axes[1].axis("off")
fig.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

In [ ]:
# Interactive pair slider (sequential bag read per step — slow on USB for large indices)
import ipywidgets as widgets
from ipywidgets import interact

n_pairs = len(camera_color_idx)


@interact(pair_idx=widgets.IntSlider(min=0, max=max(0, n_pairs - 1), step=1, value=PAIR_IDX))
def _scrub(pair_idx: int):
    cidx = int(camera_color_idx[pair_idx])
    didx = int(camera_depth_idx[pair_idx])
    if cidx < 0:
        print(f"pair {pair_idx}: no camera match")
        return
    print(
        f"pair={pair_idx} radar_t={radar_t[pair_idx]:.3f} "
        f"camera_delta={camera_delta_ms[pair_idx]:.1f} ms"
    )
    color = read_image_at_index(CAMERA_BAG, DEFAULT_COLOR_TOPIC, cidx)
    depth = read_image_at_index(CAMERA_BAG, DEFAULT_DEPTH_TOPIC, didx)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(rgb_to_display(color.data))
    axes[0].set_title(f"RGB #{cidx}")
    axes[0].axis("off")
    im = axes[1].imshow(depth_to_display(depth.data, max_m=DEPTH_MAX_M), cmap="turbo")
    axes[1].set_title(f"Depth #{didx}")
    axes[1].axis("off")
    fig.colorbar(im, ax=axes[1], fraction=0.046)
    plt.tight_layout()
    plt.show()